# Averaged across problems (≥ cutoff): First leave after good reversals (robust merge + diagnostics)

This version adds:
- a **robust merge** that guarantees the merged structure is `dict[subj] -> dict(counts)`  
- a **diagnostic cell** that prints any subjects whose merged value is not a dict, so you can see what’s coming out of `get_first_leave_after_good_reversals`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.behavior_import.import_data import *
from src.behavior_import.extract_trials import *
from src.behavior_analysis.get_good_reversal_info import *
from src.behavior_analysis.get_first_leave_after_good_reversals import *
from src.behavior_analysis.split_early_late_good_reversals import *
from src.behavior_analysis.get_diagnostic_p_value import *
from src.behavior_visualization.plot_first_leave_after_good_reversals import *
from fix_grid_maze_cohort_02_problems import *


## Parameters

In [2]:
task = "grid-maze"
# task = "open-field"

CUTOFF_PROBLEM = 1
OUT_BASE = Path("../results/figures")


## Load data and group into problems

In [3]:
folder_name = None
cohort = None
if task == "grid-maze":
    cohort = "cohort-02"
    folder_name = "3x3_maze_blocked_reward_bandit"
elif task == "open-field":
    cohort = "cohort-01"
    folder_name = "3x3_field_blocked_reward_bandit"

root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"

subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)

if task == "grid-maze" and cohort == "cohort-02":
    subjects_trials_by_problem = fix_grid_maze_cohort_02_problems(subjects_trials_by_problem)

problem_numbers = sorted(subjects_trials_by_problem.keys())
probs_used = [p for p in problem_numbers if p >= CUTOFF_PROBLEM]
print("Problems found:", problem_numbers)
print("Averaging problems >=", CUTOFF_PROBLEM, ":", probs_used)


[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-27_date-20260124/behav/._MY_04_R-2026-01-24-124422.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-55_date-20260209/behav/._MY_04_R-2026-02-09-103918.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-28_date-20260124/behav/._MY_04_R-2026-01-24-165301.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-12_date-20260116/behav/._MY_04_R-2026-01-16-143640.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volum

## Robust merge helpers

In [4]:
# Robust merge helpers (now in src)
from src.behavior_analysis.merge_across_problems import (
    sum_count_dicts,
    normalize_subject_value,
    merge_first_leave_dicts,
)

## Compute per-problem first-leave dicts (All / Early / Late)

In [5]:
first_leave_by_problem_all = {}
first_leave_by_problem_early = {}
first_leave_by_problem_late = {}

for p in probs_used:
    subjects_trials = subjects_trials_by_problem[p]
    rw = get_good_reversal_info(subjects_trials, include_first_block=False)

    first_leave_by_problem_all[p] = get_first_leave_after_good_reversals(rw)

    early, late = split_good_reversals_early_late(rw, first_n=2)
    first_leave_by_problem_early[p] = get_first_leave_after_good_reversals(early)
    first_leave_by_problem_late[p] = get_first_leave_after_good_reversals(late)

# quick diagnostic on raw output types
example_p = probs_used[0] if probs_used else None
if example_p is not None:
    ex = first_leave_by_problem_all[example_p]
    types = {}
    for s, v in ex.items():
        types[type(v).__name__] = types.get(type(v).__name__, 0) + 1
    print(f"Raw first_leave types in problem {example_p}:", types)


[SKIP] MY_05_L boundary@401 (good_reversals, block 8): more than 1 zero rewards in [0, 4, 0]
[SKIP] MY_05_L boundary@146 (good_reversals, block 4): more than 1 zero rewards in [0, 4, 0]
[SKIP] MY_04_L boundary@103 (good_reversals, block 3): more than 1 zero rewards in [0, 0, 4]
[SKIP] MY_04_N boundary@186 (good_reversals, block 7): more than 1 zero rewards in [0, 4, 0]
Raw first_leave types in problem 1: {'dict': 6}


## Merge across problems and validate structure

In [6]:
first_leave_merged_all = merge_first_leave_dicts(first_leave_by_problem_all)

# validate
bad_subjects = [(s, type(v), v) for s, v in first_leave_merged_all.items() if not isinstance(v, dict)]
print("Merged subjects:", len(first_leave_merged_all))
print("Non-dict merged values:", len(bad_subjects))
if bad_subjects:
    print("First few bad subjects:", bad_subjects[:5])

# also validate required keys
missing_keys = [(s, v.keys()) for s, v in first_leave_merged_all.items()
                if isinstance(v, dict) and not all(k in v for k in ("new_best","third","total"))]
print("Subjects missing required keys:", len(missing_keys))
if missing_keys:
    print("First few missing:", missing_keys[:5])


Merged subjects: 6
Non-dict merged values: 0
Subjects missing required keys: 0


## Stats/p-values + plots (Averaged)

In [7]:
# --- ALL ---
mean, se, n_subjects = average_first_leave_across_subjects(first_leave_merged_all)
p_value = pvalue_paired_t_new_vs_third(first_leave_merged_all, alternative="greater")
save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "First Leave After Good Reversals (Averaged)"
plot_first_leave_after_good_reversals(mean, se, first_leave_merged_all, p_value, save_path=save_path)

# --- EARLY ---
first_leave_merged_early = merge_first_leave_dicts(first_leave_by_problem_early)
mean_early, se_early, n_subjects_early = average_first_leave_across_subjects(first_leave_merged_early)
p_value_early = pvalue_paired_t_new_vs_third(first_leave_merged_early, alternative="greater")
save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "First Leave After Good Reversals (Averaged) (Early)"
plot_first_leave_after_good_reversals(mean_early, se_early, first_leave_merged_early, p_value_early, save_path=save_path)

# --- LATE ---
first_leave_merged_late = merge_first_leave_dicts(first_leave_by_problem_late)
mean_late, se_late, n_subjects_late = average_first_leave_across_subjects(first_leave_merged_late)
p_value_late = pvalue_paired_t_new_vs_third(first_leave_merged_late, alternative="greater")
save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "First Leave After Good Reversals (Averaged) (Late)"
plot_first_leave_after_good_reversals(mean_late, se_late, first_leave_merged_late, p_value_late, save_path=save_path)

print("n_subjects (all/early/late):", n_subjects, n_subjects_early, n_subjects_late)


n_subjects (all/early/late): 6 6 6
